# Empty Environment

> An empty grid environment with no walls or objects. This is useful for testing and debugging, as well as for creating custom environments by adding objects and walls to the grid.

In [ ]:
#| default_exp envs.empty

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

from fastcore.utils import patch
import numpy as np
from numpy.typing import NDArray as ndarray

from multigrid.envs.base import MultiGridEnv
from multigrid.core import Grid
from multigrid.core.constants import Direction
from multigrid.core.world_object import Goal
from multigrid.core.agent import Agent, AgentState
from multigrid.utils.obs import gen_obs_grid_encoding, gen_obs_grid_image
from multigrid.core.constants import Color


![https://i.imgur.com/wY0tT7R.gif](https://i.imgur.com/wY0tT7R.gif)

In [ ]:
#| export
class EmptyEnv(MultiGridEnv):
    """
    .. image:: https://i.imgur.com/wY0tT7R.gif
        :width: 200

    ***********
    Description
    ***********

    This environment is an empty room, and the goal for each agent is to reach the
    green goal square, which provides a sparse reward. A small penalty is subtracted
    for the number of steps to reach the goal.

    The standard setting is competitive, where agents race to the goal, and
    only the winner receives a reward.

    This environment is useful with small rooms, to validate that your RL algorithm
    works correctly, and with large rooms to experiment with sparse rewards and
    exploration. The random variants of the environment have the agents starting
    at a random position for each episode, while the regular variants have the
    agent always starting in the corner opposite to the goal.

    *************
    Mission Space
    *************

    "get to the green goal square"

    *****************
    Observation Space
    *****************

    The multi-agent observation space is a Dict mapping from agent index to
    corresponding agent observation space.

    Each agent observation is a dictionary with the following entries:

    * image : ndarray[int] of shape (view_size, view_size, :attr:`.WorldObj.dim`)
        Encoding of the agent's partially observable view of the environment,
        where the object at each grid cell is encoded as a vector:
        (:class:`.Type`, :class:`.Color`, :class:`.State`)
    * direction : int
        Agent's direction (0: right, 1: down, 2: left, 3: up)
    * mission : Mission
        Task string corresponding to the current environment configuration

    ************
    Action Space
    ************

    The multi-agent action space is a Dict mapping from agent index to
    corresponding agent action space.

    Agent actions are discrete integer values, given by:

    +-----+--------------+-----------------------------+
    | Num | Name         | Action                      |
    +=====+==============+=============================+
    | 0   | left         | Turn left                   |
    +-----+--------------+-----------------------------+
    | 1   | right        | Turn right                  |
    +-----+--------------+-----------------------------+
    | 2   | forward      | Move forward                |
    +-----+--------------+-----------------------------+
    | 3   | pickup       | Pick up an object           |
    +-----+--------------+-----------------------------+
    | 4   | drop         | Drop an object              |
    +-----+--------------+-----------------------------+
    | 5   | toggle       | Toggle / activate an object |
    +-----+--------------+-----------------------------+
    | 6   | done         | Done completing task        |
    +-----+--------------+-----------------------------+

    *******
    Rewards
    *******

    A reward of ``1 - 0.9 * (step_count / max_steps)`` is given for success,
    and ``0`` for failure.

    ***********
    Termination
    ***********

    The episode ends if any one of the following conditions is met:

    * Any agent reaches the goal
    * Timeout (see ``max_steps``)

    *************************
    Registered Configurations
    *************************

    * ``MultiGrid-Empty-5x5-v0``
    * ``MultiGrid-Empty-Random-5x5-v0``
    * ``MultiGrid-Empty-6x6-v0``
    * ``MultiGrid-Empty-Random-6x6-v0``
    * ``MultiGrid-Empty-8x8-v0``
    * ``MultiGrid-Empty-16x16-v0``
    """

    def __init__(
        self,
        size: int = 8,
        agent_start_pos: tuple[int, int] | None = (1, 1),
        agent_start_dir: Direction | None = Direction.right,
        max_steps: int | None = None,
        joint_reward: bool = False,
        success_termination_mode: str = 'any',
        **kwargs):
        """
        Parameters
        ----------
        size : int, default=8
            Width and height of the grid
        agent_start_pos : tuple[int, int], default=(1, 1)
            Starting position of the agents (random if None)
        agent_start_dir : Direction, default=Direction.right
            Starting direction of the agents (random if None)
        max_steps : int, optional
            Maximum number of steps per episode
        joint_reward : bool, default=True
            Whether all agents receive the reward when the task is completed
        success_termination_mode : 'any' or 'all', default='any'
            Whether to terminate the environment when any agent reaches the goal
            or after all agents reach the goal
        **kwargs
            See :attr:`multigrid.base.MultiGridEnv.__init__`
        """
        self.agent_start_pos = agent_start_pos
        self.agent_start_dir = agent_start_dir

        super().__init__(
            mission_space="get to the green goal square",
            grid_size=size,
            max_steps=max_steps or (4 * size**2),
            joint_reward=joint_reward,
            success_termination_mode=success_termination_mode,
            **kwargs,
        )

    def _gen_grid(self, width, height):
        pass

In [ ]:
#| export
@patch
def _gen_grid(self: EmptyEnv, width, height):
        """
        :meta private:
        """
        # Create an empty grid
        self.grid = Grid(width, height)

        # Generate the surrounding walls
        self.grid.wall_rect(0, 0, width, height)

        # Place a goal square in the bottom-right corner
        self.put_obj(Goal(), width - 2, height - 2)

        # Place the agent
        for agent in self.agents:
            if self.agent_start_pos is not None and self.agent_start_dir is not None:
                agent.state.pos = self.agent_start_pos
                agent.state.dir = self.agent_start_dir
            else:
                self.place_agent(agent)

In [ ]:
#| export
@patch
def get_goal_state(
    self: EmptyEnv,
    agent: Agent,
    agent_view_size: int,
    tile_size: int = 32,
    see_through_walls: bool = False,
) -> ndarray:
    """
    Returns the goal state RGB image for the given agent.
    The goal state is being one step away from the goal object,
    facing it.
    """
    # Find the goal object position
    goal_pos = None
    for x in range(self.grid.width):
        for y in range(self.grid.height):
            obj = self.grid.get(x, y)
            if isinstance(obj, Goal):
                goal_pos = np.array([x, y])
                break
        if goal_pos is not None:
            break

    if goal_pos is None:
        raise ValueError("No goal object found in the grid")

    # Four possible positions around the goal (right, down, left, up)
    # and the direction the agent must face to look at the goal
    candidate_positions = [
        (goal_pos + np.array([1, 0]),  Direction.left),   # agent to the right, facing left
        (goal_pos + np.array([-1, 0]), Direction.right),  # agent to the left, facing right
        (goal_pos + np.array([0, 1]),  Direction.up),     # agent below, facing up
        (goal_pos + np.array([0, -1]), Direction.down),   # agent above, facing down
    ]

    # Pick a valid candidate (inside grid, not a wall)
    goal_agent_pos = None
    goal_agent_dir = None
    for pos, dir in candidate_positions:
        x, y = pos
        if 0 <= x < self.grid.width and 0 <= y < self.grid.height:
            cell = self.grid.get(x, y)
            if cell is None or isinstance(cell, Goal):
                goal_agent_pos = pos
                goal_agent_dir = dir
                break

    if goal_agent_pos is None:
        raise ValueError("No valid position adjacent to goal found")

    # Create a temporary agent state at the goal position
    goal_agent = Agent(index=agent.index)
    goal_agent.state.pos = goal_agent_pos
    goal_agent.state.dir = goal_agent_dir
    goal_agent.state.color = agent.state.color.name

    # Compute obs grid for this goal agent state
    # We need to temporarily modify agents_states for gen_obs_grid
    original_pos = agent.state.pos
    original_dir = agent.state.dir

    agent.state.pos = goal_agent_pos
    agent.state.dir = goal_agent_dir

    # Generate the observation image
    goal_image = gen_obs_grid_image(
        self.grid,
        [goal_agent],
        self.agent_states,  # uses modified agent state
        agent_view_size,
        tile_size=tile_size,
        see_through_walls=see_through_walls,
    )[0]

    # Restore original agent state
    agent.state.pos = original_pos
    agent.state.dir = original_dir

    return goal_image

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()